In [ ]:
# ==============================================================================
# 0. PACKAGE & LIBRARY INSTALLATION ----
# ==============================================================================
options(timeout = 3600)

# Install required packages via Conda
system("conda install -c conda-forge libgdal-hdf5")
system("conda install -y r-terra=1.8_42 r-sf=1.0_20 r-gridExtra=2.3 r-ncdf4 r-fields")

# Load libraries
library(terra)
library(sf)
library(gridExtra)
library(dplyr)
library(ggplot2)
library(ncdf4)
library(jsonlite)
library(fields)
library(stringr)

# ==============================================================================
# 1. PARAMETERS AND INPUT LOADING ----
# ==============================================================================
json_data <- fromJSON("galaxy_inputs/galaxy_inputs.json")

file_copernicus <- json_data$CopernicusXDataXUser$path 
subarea         <- json_data$subarea       
area_list       <- trimws(unlist(strsplit(subarea, ",")))
multiplier      <- json_data$multiplier
index_max       <- json_data$indexXmax 
legend_pos      <- json_data$legendXposition

# ==============================================================================
# 2. LOAD NETCDF & GEBCO DATA ----
# ==============================================================================
# Read NetCDF Copernicus file
nc <- nc_open(file_copernicus)
chl_data <- ncvar_get(nc, "CHL")[, ncol(ncvar_get(nc, "CHL")):1]  # Reverse columns due to known bug
lat <- ncvar_get(nc, "latitude")
lon <- ncvar_get(nc, "longitude")
nc_close(nc)

# Create CHL raster and reproject it
chl_raster <- rast(nrows = length(lat), ncols = length(lon),
                   xmin = min(lon), xmax = max(lon),
                   ymin = min(lat), ymax = max(lat),
                   crs = "EPSG:4326")
values(chl_raster) <- as.vector(chl_data)
chl_projected <- project(chl_raster, "EPSG:6932")

# Download and load GEBCO raster
gebco_url <- "https://gis.ccamlr.org/geoserver/www/GEBCO2024_5000.tif"
download.file(gebco_url, destfile = "GEBCO2024_5000.tif", mode = "wb")
gebco_raster <- rast("GEBCO2024_5000.tif")

# ==============================================================================
# 3. HANDLE ZONES: ASD or NSWE ----
# ==============================================================================
letters_only <- str_extract(area_list, "[A-Za-z]+")
numbers_only <- str_extract(area_list, "-?\\d+")

if (is.na(letters_only[1])) {
  # --- ASD ZONE ---
  dir.create("asd", showWarnings = FALSE)
  asd_urls <- paste0("https://raw.githubusercontent.com/ccamlr/data/refs/tags/v0.5.0/geographical_data/asd/asd-shapefile-EPSG6932.", c("shp", "shx", "dbf", "prj"))
  lapply(seq_along(asd_urls), function(i) download.file(asd_urls[i], file.path("asd", basename(asd_urls[i])), mode = "wb"))

  ASD_shapes <- st_read("asd/asd-shapefile-EPSG6932.shp", quiet = TRUE)
  selected_areas <- ASD_shapes %>% filter(GAR_Short_ %in% area_list)
  area_vector <- vect(selected_areas)
  
  # Compute extent and mask both rasters
  extent_obj <- ext(mask(crop(chl_projected, area_vector), area_vector))
  gebco_masked <- mask(crop(gebco_raster, area_vector), area_vector)

} else {
  # --- NSWE ZONE ---
  bounds <- list(
    N = as.numeric(numbers_only[letters_only == "N"]),
    S = as.numeric(numbers_only[letters_only == "S"]),
    W = as.numeric(numbers_only[letters_only == "W"]),
    E = as.numeric(numbers_only[letters_only == "E"])
  )
  zone_extent <- ext(bounds$W, bounds$E, bounds$S, bounds$N)
  projected_zone <- project(rast(ext = zone_extent, crs = "EPSG:4326"), "EPSG:6932")

  extent_obj <- ext(projected_zone)
  area_vector <- NULL  # not used here
  chl_projected <- mask(crop(chl_projected, extent_obj), extent_obj)
  gebco_masked <- mask(crop(gebco_raster, extent_obj), extent_obj)
}

# Compute image dimensions
width_img <- abs(extent_obj[2] - extent_obj[1]) / 10000
height_img <- abs(extent_obj[4] - extent_obj[3]) / 10000

# ==============================================================================
# 4. FINAL DISPLAY AND EXPORT TO PNG ----
# ==============================================================================
png("outputs/Fig9.png", width = width_img * multiplier, height = height_img * multiplier)

# Define CHL color palette
chl_palette <- colorRampPalette(c(
  "#0286c4", "#1698b7", "#33a1b8", "#4bafad", "#6abbb4",
  "#82cca9", "#94dba1", "#a5e49d", "#b6eba3", "#d6f1ac",
  "#e2f9ac", "#f1feb8", "#fffdb3", "#fff2a4", "#ffe493",
  "#fed48a", "#ffcb6e", "#fcbc65", "#f8b253", "#ff9645",
  "#ff642c", "#f54d29", "#f82619", "#f40616"
))(1000)
chl_palette <- c(chl_palette, "red")
chl_breaks <- c(seq(0.03, index_max, length.out = 100), Inf)

# Define GEBCO colors
gebco_colors <- c("white", "grey")
gebco_breaks <- c(-Inf, 0, Inf)

# Plot GEBCO background
plot(gebco_masked, col = gebco_colors, breaks = gebco_breaks, legend = FALSE, axes = FALSE, box = FALSE)

# Overlay CHL data
plot(chl_projected, col = chl_palette, breaks = chl_breaks, legend = FALSE, add = TRUE)

# Add ASD boundaries and centroids if applicable
if (!is.null(area_vector)) {
  plot(area_vector, add = TRUE, lwd = 0.75 * multiplier, border = 'black')
  centroids <- st_centroid(st_geometry(selected_areas))
  text(st_coordinates(centroids), labels = selected_areas$GAR_Long_L, col = "black", cex = 1 * multiplier)
}

dev.off()


In [ ]:
# ==============================================================================
# 5. ADD LEGEND BASED ON HORIZONTAL CONFIGURATION ----
# ==============================================================================
if (horizontal == "No legend") {
  # No legend case, do nothing
} else {
  zlim <- c(0.03, 5)
  breaks_legend <- c(0.03, 1, 2, 3, 4, 5)
  labels_legend <- format(breaks_legend, nsmall = 2)
  n <- 5  # size factor

  # Path to save the legend image
  legend_file <- "outputs/Legend_Fig9.png"

  # Create the PNG file for the legend
  if (horizontal == "Horizontal") {
    png(filename = legend_file, width = 400 * n, height = 200 * n, bg = "transparent")
    par(mar = c(2, 5, 2, 2))  # margins for horizontal legend
    horizontal = TRUE
  } else {
    png(filename = legend_file, width = 200 * n, height = 400 * n, bg = "transparent")
    par(mar = c(5, 2, 2, 5))  # margins for vertical legend
    horizontal = FALSE
  }

  # Draw the legend
  image.plot(
    zlim = zlim,
    col = chl_palette,
    legend.only = TRUE,
    horizontal = horizontal,
    legend.width = 2 * n,
    legend.shrink = 0.9,
    legend.mar = 8,
    axis.args = list(
      at = breaks_legend,
      labels = labels_legend,
      cex.axis = 2,
      col.axis = "black"
    )
  )
  dev.off()
}